[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/general-sense/xdflow/blob/main/examples/notebooks/har_activity.ipynb)

# Activity recognition from phone sensors: splitting by subject

This notebook classifies six activities (walking, walking upstairs, walking downstairs, sitting, standing, lying down) from a waist-worn smartphone's accelerometer and gyroscope, using the [UCI Human Activity Recognition Using Smartphones](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones) dataset (30 volunteers, 2.56 s windows at 50 Hz that overlap by 50%).

It asks how well a simple model works on **people it has never seen**, and how much a random split over windows overstates that. It runs the analysis with xdflow and with plain scikit-learn, on the same data. The narrative version is the [HAR tutorial](https://github.com/general-sense/xdflow/blob/main/docs/tutorials/har-activity.md); the script is `examples/har_activity.py`.

Runs on a CPU in a few minutes (about 1.5 min on a laptop). The first run downloads 61 MB.

In [ ]:
%pip install -q "xdflow>=0.4"

## Load the data into one labeled array

The download is cached in `~/xdflow_data` (or `$XDFLOW_DATA_DIR`). The nine raw signal channels become one `DataArray` with `trial`, `channel` and `time` dimensions. Each window (`trial`) carries its `activity`, `subject`, the official `split` (21 train / 9 test subjects) and a `segment`: a run of consecutive windows from one subject doing one activity.

In [2]:
import os
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from xdflow import step
from xdflow.composite import Pipeline
from xdflow.core import DataContainer
from xdflow.cv import KFoldValidator, LeaveGroupOutValidator
from xdflow.transforms.basic_transforms import FlattenTransform
from xdflow.transforms.sklearn_transform import SKLearnPredictor, SKLearnTransformer

URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
ROOT = "UCI HAR Dataset"
CHANNELS = [f"{sensor}_{axis}" for sensor in ("body_acc", "body_gyro", "total_acc") for axis in "xyz"]
FS = 50  # Hz; every window is 128 samples = 2.56 s, consecutive windows overlap by 50%
BANDS = ((0, 1), (1, 3), (3, 6), (6, 12), (12, 26))  # Hz; the last band includes the 25 Hz Nyquist bin
FEATURES = ["mean", "std", "min", "max", "p25", "p75"] + [f"logpow_{lo}_{hi}hz" for lo, hi in BANDS]
SEED = 0
DATA_DIR = Path(os.environ.get("XDFLOW_DATA_DIR", "~/xdflow_data")).expanduser()

In [3]:
def check_archive(archive: Path) -> Path:
    """Fail loudly unless the dataset archive has every file the loader reads."""
    expected = {f"{ROOT}/activity_labels.txt"} | {
        f"{ROOT}/{split}/{name}_{split}.txt"
        for split in ("train", "test")
        for name in ("y", "subject", *(f"Inertial Signals/{channel}" for channel in CHANNELS))
    }
    with zipfile.ZipFile(archive) as z:
        missing = expected - set(z.namelist())
    if missing:
        raise RuntimeError(f"{archive} lacks {sorted(missing)}; delete it to download again.")
    return archive


def fetch_har(data_dir: Path) -> Path:
    """Download once and return the checked inner dataset archive."""
    archive = data_dir / f"{ROOT}.zip"
    if not archive.exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {URL} (61 MB) to {data_dir} ...")
        with tempfile.TemporaryDirectory(dir=data_dir) as tmp:  # only a complete, checked archive enters the cache
            outer = Path(tmp) / "download.zip"
            urllib.request.urlretrieve(URL, outer)
            with zipfile.ZipFile(outer) as z:
                z.extract(archive.name, tmp)
            os.replace(check_archive(Path(tmp) / archive.name), archive)
    return check_archive(archive)


def load_har(archive: Path) -> xr.DataArray:
    """(trial, channel, time) windows with activity, subject, split and segment coordinates."""
    pieces = []
    with zipfile.ZipFile(archive) as z:
        codes = dict(line.split() for line in z.read(f"{ROOT}/activity_labels.txt").decode().splitlines() if line)
        for split in ("train", "test"):
            signals = np.stack(
                [np.loadtxt(z.open(f"{ROOT}/{split}/Inertial Signals/{ch}_{split}.txt")) for ch in CHANNELS], axis=1
            )
            activity = np.loadtxt(z.open(f"{ROOT}/{split}/y_{split}.txt"), dtype=int)
            subject = np.loadtxt(z.open(f"{ROOT}/{split}/subject_{split}.txt"), dtype=int)
            if signals.shape != (len(activity), len(CHANNELS), 128) or len(subject) != len(activity):
                raise ValueError(f"{split}: inconsistent shapes {signals.shape}, {activity.shape}, {subject.shape}")
            pieces.append(
                xr.DataArray(
                    signals,
                    dims=("trial", "channel", "time"),
                    coords={
                        "channel": CHANNELS,
                        "time": np.arange(128) / FS,
                        "activity": ("trial", [codes[str(code)].lower() for code in activity]),
                        "subject": ("trial", subject),
                        "split": ("trial", np.full(len(activity), split)),
                    },
                )
            )
    data = xr.concat(pieces, dim="trial", join="exact")  # fails loudly if channels or times differ
    # A segment is a run of consecutive windows (file order) from one subject doing one activity.
    activity, subject = data["activity"].values, data["subject"].values
    new_segment = np.r_[True, (activity[1:] != activity[:-1]) | (subject[1:] != subject[:-1])]
    return data.assign_coords(trial=np.arange(data.sizes["trial"]), segment=("trial", np.cumsum(new_segment)))


data = load_har(fetch_har(DATA_DIR))
data

<xarray.DataArray (trial: 10299, channel: 9, time: 128)> Size: 95MB
array([[[ 1.808515e-04,  1.013856e-02,  9.275574e-03, ...,
         -1.147484e-03, -2.222655e-04,  1.575500e-03],
        [ 1.076681e-02,  6.579480e-03,  8.928878e-03, ...,
          1.714439e-04,  1.574181e-03,  3.070189e-03],
        [ 5.556068e-02,  5.512483e-02,  4.840473e-02, ...,
          2.647864e-03,  2.381057e-03, -2.269757e-03],
        ...,
        [ 1.012817e+00,  1.022833e+00,  1.022028e+00, ...,
          1.018445e+00,  1.019372e+00,  1.021171e+00],
        [-1.232167e-01, -1.268756e-01, -1.240037e-01, ...,
         -1.240696e-01, -1.227451e-01, -1.213260e-01],
        [ 1.029341e-01,  1.056872e-01,  1.021025e-01, ...,
          1.003852e-01,  9.987355e-02,  9.498741e-02]],

       [[ 1.093752e-03,  4.550077e-03,  2.879173e-03, ...,
         -4.646144e-03, -2.941333e-03, -1.598751e-03],
        [-4.687588e-03, -7.487894e-03, -8.429991e-03, ...,
         -5.479850e-03, -2.987481e-03, -7.192009e-03],
        [-2.685954e-02, -2.509841e-02, -2.597534e-02, ...,
         -2.996671e-04, -4.886324e-03, -3.397195e-03],
...
          1.156389e+00,  1.243857e+00,  1.323546e+00],
        [-3.971532e-01, -2.848634e-01, -2.635297e-01, ...,
         -2.283478e-01, -2.583220e-01, -3.472416e-01],
        [-3.940817e-01, -3.151097e-01, -2.139040e-01, ...,
         -3.512052e-03, -1.117857e-01, -2.760682e-01]],

       [[-1.936425e-01, -6.498738e-02, -9.712210e-02, ...,
         -5.075521e-02, -1.980675e-02, -1.104015e-02],
        [-1.907511e-01, -2.035990e-01, -2.083832e-01, ...,
         -1.047171e-01, -2.076396e-02,  5.243883e-02],
        [ 1.958357e-01, -1.531400e-01, -2.710627e-01, ...,
          1.732707e-01,  1.956384e-01,  2.184321e-01],
        ...,
        [ 7.713622e-01,  9.000949e-01,  8.681034e-01, ...,
          9.188616e-01,  9.494752e-01,  9.578348e-01],
        [-4.250499e-01, -4.375916e-01, -4.421595e-01, ...,
         -3.516799e-01, -2.675260e-01, -1.941603e-01],
        [-5.327655e-02, -4.020727e-01, -5.197379e-01, ...,
         -7.253919e-02, -5.097549e-02, -2.892477e-02]]],
      shape=(10299, 9, 128))
Coordinates:
  * trial     (trial) int64 82kB 0 1 2 3 4 5 ... 10294 10295 10296 10297 10298
    activity  (trial) <U18 742kB 'standing' 'standing' ... 'walking_upstairs'
    subject   (trial) int64 82kB 1 1 1 1 1 1 1 1 1 ... 24 24 24 24 24 24 24 24
    split     (trial) <U5 206kB 'train' 'train' 'train' ... 'test' 'test' 'test'
    segment   (trial) int64 82kB 1 1 1 1 1 1 1 1 ... 400 400 400 400 400 400 400
  * channel   (channel) <U11 396B 'body_acc_x' 'body_acc_y' ... 'total_acc_z'
  * time      (time) float64 1kB 0.0 0.02 0.04 0.06 0.08 ... 2.48 2.5 2.52 2.54

Classes are roughly balanced. Always predicting the most common class (`laying`) is the baseline to beat.

In [4]:
counts = pd.crosstab(data["activity"].values, data["split"].values, rownames=["activity"], colnames=["split"])
print(counts.to_string())
print("Majority-class accuracy:", (counts.max() / counts.sum()).round(3).to_dict())

split               test  train
activity                       
laying               537   1407
sitting              491   1286
standing             532   1374
walking              496   1226
walking_downstairs   420    986
walking_upstairs     471   1073
Majority-class accuracy: {'test': 0.182, 'train': 0.191}


## Features and pipeline

Per channel and window: six summary statistics and the log power in five frequency bands, so 9 × 11 = 99 features. The features are one NumPy function, which the scikit-learn version below calls too. In xdflow, `@step` makes it a pipeline step. Each window's features depend on that window alone (nothing is learned from data), so validators compute them once, before the folds. The scaler and the classifier are stateful and are refit inside every training fold.

In [5]:
def window_features(x: np.ndarray, fs: float) -> np.ndarray:
    """Summary statistics and log band power over the last axis (time); returns (..., len(FEATURES))."""
    power = np.abs(np.fft.rfft(x - x.mean(axis=-1, keepdims=True), axis=-1)) ** 2
    freqs = np.fft.rfftfreq(x.shape[-1], d=1 / fs)
    band_power = [power[..., (freqs >= lo) & (freqs < hi)].sum(axis=-1) for lo, hi in BANDS]
    stats = [x.mean(-1), x.std(-1), x.min(-1), x.max(-1), *np.percentile(x, [25, 75], axis=-1)]
    return np.stack(stats + [np.log(p + 1e-10) for p in band_power], axis=-1)


@step
def features(data: xr.DataArray, fs: float = FS) -> xr.DataArray:
    """Per-channel window features. Each window's features depend on that window only, so cross-validation
    computes them once for all folds."""
    if not np.allclose(np.diff(data["time"].values), 1 / fs):
        raise ValueError(f"The time coordinate is not sampled at fs={fs} Hz.")
    # apply_ufunc replaces the time dim with a feature dim and keeps every trial and channel coordinate.
    return xr.apply_ufunc(
        window_features, data, input_core_dims=[["time"]], output_core_dims=[["feature"]], kwargs={"fs": fs}
    ).assign_coords(feature=FEATURES)


def build_pipeline() -> Pipeline:
    return Pipeline(
        name="har",
        steps=[
            ("features", features(fs=FS)),  # stateless: computed once, before the folds
            ("flatten", FlattenTransform(dims=("channel", "feature"))),
            ("scale", SKLearnTransformer(StandardScaler, sample_dim="trial")),  # stateful: refit per fold
            (
                "classifier",
                SKLearnPredictor(LogisticRegression, sample_dim="trial", target_coord="activity", max_iter=2000),
            ),
        ],
    )

## The split boundary is a coordinate name

Cross-validation uses the 21 official training subjects only. The three 5-fold validators differ only in what they keep apart: nothing (windows shuffled, stratified by activity), `segment`, or `subject`.

`LeaveGroupOutValidator` keeps each group on one side of every split, which is the right tool for questions about unseen subjects. (`GroupedKFoldValidator` serves per-group models: it puts every group in every fold.)

In [6]:
common = {"scoring": "accuracy", "verbose": False}
train = DataContainer(data.isel(trial=data["split"].values == "train"))  # the 21 official training subjects
validators = {
    "windows": KFoldValidator(n_splits=5, shuffle=True, random_state=SEED, **common),
    "segments": LeaveGroupOutValidator(group_coord="segment", n_splits=5, random_state=SEED, **common),
    "subjects": LeaveGroupOutValidator(group_coord="subject", n_splits=5, random_state=SEED, **common),
}
xdf = {}
for name, cv in validators.items():
    cv.set_pipeline(build_pipeline())
    xdf[name] = cv.cross_validate(train)
    print(f"5-fold, split unit = {name}: {xdf[name]:.3f}")

5-fold, split unit = windows: 0.972


5-fold, split unit = segments: 0.949


5-fold, split unit = subjects: 0.914


Then leave-one-subject-out on the training subjects, and a final score on the 9 official test subjects, which take no part in fitting.

In [7]:
test_subjects = np.unique(data["subject"].values[data["split"].values == "test"]).tolist()
loso = LeaveGroupOutValidator(group_coord="subject", test_group_ids=test_subjects, **common)
loso.set_pipeline(build_pipeline())
xdf["loso"] = loso.cross_validate(DataContainer(data))
xdf["holdout"] = loso.score_on_holdout(DataContainer(data))
print(f"Leave one subject out: {xdf['loso']:.3f}, holdout on {test_subjects}: {xdf['holdout']:.3f}")

Leave one subject out: 0.912, holdout on [2, 4, 9, 10, 12, 13, 18, 20, 24]: 0.922


## Predictions keep their metadata

Out-of-fold predictions come back with every trial coordinate plus a `fold` coordinate, so per-subject accuracy and a confusion table are a `groupby` / `crosstab` away. With one subject per fold, the mean of the per-subject accuracies is the leave-one-subject-out score.

In [8]:
oof, held = loso.oof_predictions_container_.data, loso.holdout_container_.data
per_subject_cv = (oof == oof["activity"]).groupby("subject").mean().to_series()
per_subject_holdout = (held == held["activity"]).groupby("subject").mean().to_series()
assert np.isclose(per_subject_cv.mean(), xdf["loso"])
print("Training subjects (LOSO):", per_subject_cv.round(2).to_dict())
print("Test subjects (holdout): ", per_subject_holdout.round(2).to_dict())
print(f"LOSO across subjects: mean {per_subject_cv.mean():.3f}, SD {per_subject_cv.std():.3f}")
oof

Training subjects (LOSO): {1: 0.99, 3: 0.9, 5: 0.88, 6: 0.87, 7: 0.92, 8: 0.89, 11: 0.99, 14: 0.66, 15: 0.99, 16: 0.73, 17: 0.97, 19: 0.87, 21: 0.96, 22: 0.92, 23: 0.95, 25: 0.89, 26: 0.95, 27: 0.99, 28: 0.9, 29: 0.96, 30: 0.98}
Test subjects (holdout):  {2: 0.96, 4: 0.96, 9: 0.82, 10: 0.75, 12: 0.92, 13: 0.94, 18: 0.95, 20: 0.95, 24: 0.99}
LOSO across subjects: mean 0.912, SD 0.085


<xarray.DataArray 'prediction' (trial: 7352)> Size: 529kB
array(['standing', 'standing', 'standing', ..., 'walking_upstairs',
       'walking_upstairs', 'walking_upstairs'],
      shape=(7352,), dtype='<U18')
Coordinates:
  * trial     (trial) int64 59kB 0 1 2 3 4 5 6 ... 7346 7347 7348 7349 7350 7351
    subject   (trial) int64 59kB 1 1 1 1 1 1 1 1 1 ... 30 30 30 30 30 30 30 30
    split     (trial) <U5 147kB 'train' 'train' 'train' ... 'train' 'train'
    segment   (trial) int64 59kB 1 1 1 1 1 1 1 1 ... 280 280 280 280 280 280 280
    activity  (trial) <U18 529kB 'standing' 'standing' ... 'walking_upstairs'
    fold      (trial) int64 59kB 0 0 0 0 0 0 0 0 0 ... 20 20 20 20 20 20 20 20
Attributes:
    data_history:  [{'class': 'features', 'params': {'sel': None, 'drop_sel':...

In [9]:
confusion = pd.crosstab(
    oof["activity"].values, oof.values, normalize="index", rownames=["true"], colnames=["predicted"]
)
(100 * confusion).round(1)

predicted,laying,sitting,standing,walking,walking_downstairs,walking_upstairs
true,,,,,,
laying,98.0,0.0,0.2,0.0,0.1,1.6
sitting,0.9,92.0,7.1,0.0,0.0,0.1
standing,0.0,6.9,92.9,0.1,0.0,0.0
walking,0.0,0.0,0.0,88.0,6.0,6.0
walking_downstairs,0.0,0.1,0.0,6.3,85.0,8.6
walking_upstairs,0.0,0.0,0.0,3.4,7.3,89.4


## The same analysis in plain scikit-learn

Same features (same function), same windows in the same order. Grouped CV is one `groups=` argument; the groups array just has to be built and kept aligned with `X`.

In [10]:
SKLEARN_MODELS = {
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-nearest neighbours (k=5)": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "random forest (100 trees)": lambda: RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
}

X = window_features(data.transpose("trial", "channel", "time").values, FS).reshape(data.sizes["trial"], -1)
y, subjects, segments = data["activity"].values, data["subject"].values, data["segment"].values
is_train = data["split"].values == "train"
splits = {
    "windows": (StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), None),
    "segments": (GroupKFold(n_splits=5), segments[is_train]),
    "subjects": (GroupKFold(n_splits=5), subjects[is_train]),
    "loso": (LeaveOneGroupOut(), subjects[is_train]),
}


def run_sklearn(model_name: str, split_names: tuple[str, ...]) -> dict:
    model = SKLEARN_MODELS[model_name]
    scores = {
        name: cross_val_score(model(), X[is_train], y[is_train], cv=splits[name][0], groups=splits[name][1]).mean()
        for name in split_names
    }
    scores["holdout"] = model().fit(X[is_train], y[is_train]).score(X[~is_train], y[~is_train])
    return scores


skl = run_sklearn("logistic regression", ("windows", "segments", "subjects", "loso"))
pd.DataFrame({"xdflow": xdf, "sklearn": skl}).round(3)

,xdflow,sklearn
windows,0.972,0.972
segments,0.949,0.943
subjects,0.914,0.916
loso,0.912,0.912
holdout,0.922,0.922


Where the splits are identical (shuffled 5-fold, leave-one-subject-out, holdout) the two columns match exactly. The grouped 5-fold rows agree within 0.006.

## Does the inflation depend on the model?

In [11]:
by_model = {"logistic regression": skl}
by_model |= {
    name: run_sklearn(name, ("windows", "subjects")) for name in SKLEARN_MODELS if name != "logistic regression"
}
pd.DataFrame(by_model).T[["windows", "subjects", "holdout"]].round(3)

,windows,subjects,holdout
logistic regression,0.972,0.916,0.922
k-nearest neighbours (k=5),0.966,0.869,0.875
random forest (100 trees),0.981,0.895,0.893


## What this shows

- **Same model, same numbers:** xdflow and scikit-learn agree on the same splits.
- **The split decides the answer:** shuffled windows report 0.972 and pick the wrong model; keeping subjects apart gives 0.914, in line with the 0.922 on never-seen people. In xdflow that is one argument, `group_coord="subject"`.
- **Per-subject accuracy and confusion tables come straight from the data**, because predictions keep `subject` and `activity` attached.

See the [HAR tutorial](https://github.com/general-sense/xdflow/blob/main/docs/tutorials/har-activity.md) for the full results.